In [1]:
from main import D2TAgentExperimentRunner
from agents.llm_model import UnifiedModel, model_name

model_provider = "openai" #ollama, openai, hf, aixplain
language = "en"

runner = D2TAgentExperimentRunner(
    provider=model_provider,
    language=language,
    dataset_path="data/D2T-1-FA_same3to6_min50_max500_sample.xml",
    output_dir="results_debug_ga",
)
# runner.show_workflow_graph(workflow = "default", xray=True,)

/home/chinonso/anaconda3/envs/lang2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-03 13:48:31.572394: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-03 13:48:31.598623: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 13:48:32.120660: I tensorflow/core/util/port.cc:153] oneDNN custom operatio

Extracted 10 modified triplesets from data/D2T-1-FA_same3to6_min50_max500_sample.xml.
Building workflows with provider=openai, language=en.
Workflows built: ['default', 'unified_worker', 'single_module', 'no_guardrail', 'no_finalizer', 'no_orchestrator']


In [2]:
# Inspect how many samples and triple sizes
data, num_samples, triple_lengths = runner.inspect_data
print("Num samples:", num_samples)
print("Triple lengths:", triple_lengths)
sample = 1

Num samples: 10
Triple lengths: [{1: 156}, {2: 252}, {3: 149}, {4: 130}, {5: 174}, {6: 209}, {7: 201}, {8: 219}, {9: 204}, {10: 140}]


In [3]:
# data[sample - 1]

In [4]:
# # 1. Debug on 5 samples with the default architecture
# state = runner.run_sample(sample_id=sample, workflow="default", save=True)

# 2. Debug with an ablation, for example no_orchestrator
#result =  runner.run_sample(sample_id=2, workflow="no_orchestrator")

# # New unified worker architecture
state = runner.run_sample(sample_id=1, workflow="unified_worker", save=True)
# print(state.get("final_response", ""))

Running workflow='unified_worker' on sample_id=1 (index=0).


> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Final Answer",
  "action_input": [
    ["Amsterdam", "country", "Netherlands"],
    ["Amsterdam", "type", "Capital_of_the_Netherlands"],
    ["Amsterdam", "type", "Municipalities_of_the_Netherlands"],
    ["Amsterdam", "description", "پڕ گەلھەترین شار و پایتەختی ھۆڵەند"],
    ["Amsterdam", "description", "Alankomaiden pääkaupunki"],
    ["Amsterdam", "description", "Haaptstad vun Holland"],
    ["Amsterdam", "description", "Hollanda'nın başkenti ve en büyük şehri"],
    ["Amsterdam", "areaTotal", "219.32"],
    ["Amsterdam", "areaLand", "165760000.0"],
    ["Amsterdam", "areaWater", "53560000.0"],
    ["Amsterdam", "elevation", "-2.0"],
    ["Amsterdam", "populationTotal", "933680"],
    ["Amsterdam", "populationUrban", "1477213"],
    ["Amsterdam", "demonym", "Amsterdammer"],
    ["Amsterdam", "motto", "Heldhaftig, Vastberaden, Barmhartig(Valiant, Steadfast, C

In [5]:
generated_text = state.get("final_response", "")
prefix = "final answer:"
if generated_text.lower().startswith(prefix):
    generated_text = generated_text[len(prefix):].lstrip()
print(generated_text)

Amsterdam is the capital and one of the municipalities of the Netherlands. It is described as the most populous city and the capital of the Netherlands in several languages, including Kurdish, Finnish, German, and Turkish.

The city covers a total area of 219.32 square kilometers, with 165.76 square kilometers of land and 53.56 square kilometers of water, and lies at an elevation of -2.0 meters. Amsterdam has a total population of 933,680, with an urban population of 1,477,213. Residents are known as Amsterdammers. The city's motto is "Heldhaftig, Vastberaden, Barmhartig" (Valiant, Steadfast, Compassionate). Amsterdam's postal codes range from 1000 to 1183, and the area code is 020. The city operates on Central European Time (UTC+1) and observes Central European Summer Time (UTC+2).

Amsterdam is governed by a municipal council and includes subdivisions such as Amsterdam-Centrum. The city is also home to the Academic Medical Center. It serves regions such as the Official Museums of Ams

In [6]:
uc.ie

NameError: name 'uc' is not defined

In [ ]:
# # 1. End to end single shot generation (no multi agent pipeline)
# e2e_result = runner.run_end_to_end(
#     sample_id=sample,
#     provider=model_provider,
#     temperature=0.0,
# )
# print("=== End to end generated text ===")
# print(e2e_result["generated_text"])

=== End to end generated text ===
Is í Amsterdam príomhchathair na hÍsiltíre agus an chathair is mó sa tír sin. Tá sí suite i dtír na Ísiltíre agus tugtar "Amsterdammer" ar a muintir. Tá réimse leathan cur síos idirnáisiúnta uirthi, mar shampla: "پڕ گەلھەترین شار و پایتەختی ھۆڵەند", "Alankomaiden pääkaupunki", "Haaptstad vun Holland", agus "Hollanda'nın başkenti ve en büyük şehri". Tá sí aitheanta go hoifigiúil mar cheann de na "Municipalities of the Netherlands" agus mar "Capital of the Netherlands".

Tá achar iomlán 219.32 ciliméadar cearnach ag Amsterdam, le 165,760,000 méadar cearnach de thalamh agus 53,560,000 méadar cearnach de huisce. Tá sí suite ag airde -2.0 méadar faoi leibhéal na farraige. Tá an chathair faoi bhainistíocht an Municipal council (Netherlands). Is é "Heldhaftig, Vastberaden, Barmhartig (Valiant, Steadfast, Compassionate)" mana oifigiúil na cathrach. Tá daonra iomlán 933,680 aici, agus sroicheann an daonra uirbeach 1,477,213. Is é 020 an cód gutháin agus tá an r

In [ ]:
generated_text = state.get("final_response", "")
prefix = "final answer:"
if generated_text.lower().startswith(prefix):
    generated_text = generated_text[len(prefix):].lstrip()
print(generated_text)

Is í Amsterdam príomhchathair na hÍsiltíre agus tá sí ar cheann de na bardais sa tír. Tugtar 'Amsterdammer' ar a muintir agus is é a mana ná "Heldhaftig, Vastberaden, Barmhartig" (Cróga, Dílse, Trócaireach). Tá cur síos uirthi mar "پڕ گەلھەترین شار و پایتەختی ھۆڵەند", "Alankomaiden pääkaupunki", "Haaptstad vun Holland", agus "Hollanda'nın başkenti ve en büyük şehri". Tá sí suite san Ísiltír agus is í an chathair is mó sa tír freisin. Tá an chathair faoi bhainistíocht an Municipal council (Netherlands) agus tá fo-roinn aici darb ainm Amsterdam-Centrum.

Tá achar iomlán 219.32 ciliméadar cearnach ag Amsterdam, le 165,760,000 méadar cearnach de thalamh agus 53,560,000 méadar cearnach de uisce. Tá sí suite 2 mhéadar faoi leibhéal na farraige. Tá na cóid phoist idir 1000 agus 1183, agus is é 020 an cód gutháin. Tá sí sa chrios ama Central European Time agus Central European Summer Time, le haistriú UTC +1 agus +2. Tá daonra iomlán 933,680 aici, agus tá 1,477,213 duine sa limistéar uirbeach.

In [ ]:
uc.i

In [ ]:
generated_text = state.get("final_response", "")
print(generated_text)

Final Answer: Amsterdam is both a municipality and the capital of the Netherlands. Its residents are known as Amsterdammers, and the city’s motto is "Heldhaftig, Vastberaden, Barmhartig," which translates to "Valiant, Steadfast, Compassionate." Amsterdam is described in various languages as "پڕ گەلھەترین شار و پایتەختی ھۆڵەند," "Alankomaiden pääkaupunki," "Haaptstad vun Holland," and "Hollanda'nın başkenti ve en büyük şehri," all emphasizing its status as the capital and largest city of the country.

The city covers a total area of 219.32 square kilometers, with 165.76 square kilometers of land and 53.56 square kilometers of water. Amsterdam sits at an elevation of -2.0 meters and includes subdivisions such as Amsterdam-Centrum. The total population is 933,680, while the urban area is home to 1,477,213 people. The city is governed by the municipal council of the Netherlands, uses the area code 020, and has postal codes ranging from 1000 to 1183. Amsterdam observes Central European Time

In [ ]:
data[sample -1]

[['Amsterdam', 'areaTotal', '219.32'],
 ['Amsterdam', 'areaCode', '020'],
 ['Amsterdam', 'areaLand', '165760000.0'],
 ['Amsterdam', 'areaWater', '53560000.0'],
 ['Amsterdam', 'country', 'Netherlands'],
 ['Amsterdam', 'demonym', 'Amsterdammer'],
 ['Amsterdam', 'description', 'پڕ گەلھەترین شار و پایتەختی ھۆڵەند'],
 ['Amsterdam', 'description', 'Alankomaiden pääkaupunki'],
 ['Amsterdam', 'description', 'Haaptstad vun Holland'],
 ['Amsterdam', 'description', "Hollanda'nın başkenti ve en büyük şehri"],
 ['Amsterdam', 'elevation', '-2.0'],
 ['Amsterdam', 'governingBody', 'Municipal_council_(Netherlands)'],
 ['Amsterdam',
  'motto',
  'Heldhaftig, Vastberaden, Barmhartig(Valiant, Steadfast, Compassionate)'],
 ['Amsterdam', 'populationTotal', '933680'],
 ['Amsterdam', 'populationUrban', '1477213'],
 ['Amsterdam', 'postalCode', '1000–1183'],
 ['Amsterdam', 'subdivision', 'Amsterdam-Centrum'],
 ['Amsterdam', 'timeZone', 'Central_European_Summer_Time'],
 ['Amsterdam', 'timeZone', 'Central_Europea

In [ ]:
# import json
# import os
# from main import D2TAgentExperimentRunner
# # form agents.llm_model import UnifiedModel, model_name # Uncomment if needed in your environment

# def clean_text(text):
#     """
#     Cleans the generated text based on the user's logic.
#     Removes 'final answer:' prefix if present.
#     """
#     if not text:
#         return ""
#     prefix = "final answer:"
#     if text.lower().startswith(prefix):
#         text = text[len(prefix):].lstrip()
#     return text

# def run_batch_experiments():
#     # Configuration
#     model_provider = "openai" # ollama, openai, hf, aixplain
#     dataset_path = "data/D2T-1-FA_same3to6_min50_max500_sample.xml"
#     output_dir_base = "results_debug"
    
#     # We want to run for both English and Irish (Gaeilge)
#     languages = ["en", "ga"]
    
#     # How many samples to run? Set to None to run all samples found in the dataset
#     # Or set to an integer (e.g., 5) for debugging
#     max_samples_to_run = None 

#     for lang in languages:
#         print(f"\n{'='*30}")
#         print(f"Starting experiments for Language: {lang.upper()}")
#         print(f"{'='*30}")

#         # 1. Initialize the Runner for the specific language
#         runner = D2TAgentExperimentRunner(
#             provider=model_provider,
#             language=lang,
#             dataset_path=dataset_path,
#             output_dir=f"{output_dir_base}_{lang}",
#         )

#         # 2. Inspect data to get the list of samples
#         data, num_samples, triple_lengths = runner.inspect_data
#         print(f"Total samples available: {num_samples}")

#         # Determine how many to process
#         limit = num_samples
#         if max_samples_to_run is not None and max_samples_to_run < num_samples:
#             limit = max_samples_to_run
        
#         # Prepare lists to store results
#         multi_agent_results = []
#         e2e_results = []

#         # 3. Iterate through samples
#         for i in range(1, limit + 1):
#             print(f"Processing sample {i}/{limit}...")
            
#             # Get the input triples from the raw data
#             # Note: data is 0-indexed, sample_id is 1-indexed usually in these runners
#             current_triples = data[i - 1] 

#             # --- A. Multi-Agent Workflow (Default) ---
#             try:
#                 state = runner.run_sample(sample_id=i, workflow="default", save=True)
#                 raw_ma_text = state.get("final_response", "")
#                 cleaned_ma_text = clean_text(raw_ma_text)
                
#                 multi_agent_results.append({
#                     "index": i,
#                     "triples": current_triples,
#                     "output": cleaned_ma_text
#                 })
#             except Exception as e:
#                 print(f"Error running Multi-Agent on sample {i}: {e}")
#                 multi_agent_results.append({
#                     "index": i,
#                     "triples": current_triples,
#                     "output": "ERROR"
#                 })

#             # --- B. End-to-End Workflow ---
#             try:
#                 e2e_result = runner.run_end_to_end(
#                     sample_id=i,
#                     provider=model_provider,
#                     temperature=0.0,
#                 )
#                 raw_e2e_text = e2e_result.get("generated_text", "")
#                 # Assuming E2E might need the same cleaning, though usually it's direct.
#                 # Applying cleaning just in case models output prefixes.
#                 cleaned_e2e_text = clean_text(raw_e2e_text)

#                 e2e_results.append({
#                     "index": i,
#                     "triples": current_triples,
#                     "output": cleaned_e2e_text
#                 })
#             except Exception as e:
#                 print(f"Error running E2E on sample {i}: {e}")
#                 e2e_results.append({
#                     "index": i,
#                     "triples": current_triples,
#                     "output": "ERROR"
#                 })

#         # 4. Save to JSON files
#         # Map 'en' to 'EN' and 'ga' to 'GA' for filenames
#         suffix = lang.upper()
        
#         ma_filename = f"pilot_multi_agent_{suffix}.json"
#         e2e_filename = f"pilot_e2e_{suffix}.json"

#         print(f"Saving {len(multi_agent_results)} items to {ma_filename}...")
#         with open(ma_filename, 'w', encoding='utf-8') as f:
#             json.dump(multi_agent_results, f, indent=4, ensure_ascii=False)

#         print(f"Saving {len(e2e_results)} items to {e2e_filename}...")
#         with open(e2e_filename, 'w', encoding='utf-8') as f:
#             json.dump(e2e_results, f, indent=4, ensure_ascii=False)

#     print("\nAll experiments completed.")

# if __name__ == "__main__":
#     run_batch_experiments()


Starting experiments for Language: EN
Extracted 10 modified triplesets from data/D2T-1-FA_same3to6_min50_max500_sample.xml.
Building workflows with provider=openai, language=en.
Workflows built: ['default', 'single_module', 'no_guardrail', 'no_finalizer', 'no_orchestrator']
Total samples available: 10
Processing sample 1/10...
Running workflow='default' on sample_id=1 (index=0).


> Entering new AgentExecutor chain...
Could not parse LLM output: Action:{
  "action": "Final Answer",
  "action_input": [
    ["Amsterdam", "country", "Netherlands"],
    ["Amsterdam", "type", "Capital_of_the_Netherlands"],
    ["Amsterdam", "type", "Municipalities_of_the_Netherlands"],
    ["Amsterdam", "demonym", "Amsterdammer"],
    ["Amsterdam", "motto", "Heldhaftig, Vastberaden, Barmhartig(Valiant, Steadfast, Compassionate)"],
    ["Amsterdam", "governingBody", "Municipal_council_(Netherlands)"],
    ["Amsterdam", "subdivision", "Amsterdam-Centrum"],
    ["Amsterdam", "areaTotal", "219.32"],
    ["Amst

In [ ]:
# import json
# import os

# from main import D2TAgentExperimentRunner

# # Outputs that should trigger a rerun
# INVALID_OUTPUTS = {
#     "ERROR",
#     "NO SURFACE REALIZATION OUTPUT AVAILABLE.",
# }

# DATASET_PATH = "data/D2T-1-FA_same3to6_min50_max500_sample.xml"
# MODEL_PROVIDER = "openai"  # or "ollama", "hf", "aixplain"
# OUTPUT_DIR_BASE = "results_debug"  # same base as before, runner will handle suffix


# def clean_text(text: str) -> str:
#     """
#     Remove 'final answer:' prefix if present. Return normalized text.
#     """
#     if not text:
#         return ""
#     prefix = "final answer:"
#     lower = text.lower()
#     if lower.startswith(prefix):
#         text = text[len(prefix):].lstrip()
#     return text


# def repair_file_for_language(lang: str) -> None:
#     """
#     For a given language ('en' or 'ga'):
#       - load pilot_multi_agent_XX.json and pilot_e2e_XX.json
#       - rerun only samples with INVALID_OUTPUTS
#       - reindex all items to be zero based
#       - overwrite the JSON files
#     """
#     suffix = lang.upper()
#     ma_filename = f"pilot_multi_agent_{suffix}.json"
#     e2e_filename = f"pilot_e2e_{suffix}.json"

#     if not os.path.exists(ma_filename) or not os.path.exists(e2e_filename):
#         print(f"[{lang}] One or both JSON files not found. Skipping.")
#         return

#     print(f"\n========== Language: {lang.upper()} ==========")

#     # Initialise runner for this language once
#     runner = D2TAgentExperimentRunner(
#         provider=MODEL_PROVIDER,
#         language=lang,
#         dataset_path=DATASET_PATH,
#         output_dir=f"{OUTPUT_DIR_BASE}_{lang}",
#     )
#     data, num_samples, _ = runner.inspect_data
#     print(f"Dataset has {num_samples} samples.")

#     # ---------- Multi agent file ----------
#     print(f"[{lang}] Loading {ma_filename}...")
#     with open(ma_filename, "r", encoding="utf-8") as f:
#         multi_agent_results = json.load(f)

#     print(f"[{lang}] Multi agent entries: {len(multi_agent_results)}")

#     fixed_ma = 0
#     for pos, item in enumerate(multi_agent_results):
#         original_index = item.get("index", pos)
#         # Original script used 1 based index in the file
#         # but we will trust the position and use sample_id = pos + 1
#         sample_id = pos + 1

#         # Safety: if you want to double check alignment, you can uncomment:
#         # if original_index != sample_id:
#         #     print(f"[MA][{lang}] Warning: original_index {original_index} "
#         #           f"!= sample_id {sample_id}")

#         output = item.get("output", "")
#         if output in INVALID_OUTPUTS:
#             print(
#                 f"[MA][{lang}] Re-running sample_id={sample_id} "
#                 f"(old index={original_index}) due to output='{output}'"
#             )
#             try:
#                 state = runner.run_sample(
#                     sample_id=sample_id,
#                     workflow="default",
#                     save=True,
#                 )
#                 raw_ma_text = state.get("final_response", "")
#                 cleaned_ma_text = clean_text(raw_ma_text)
#                 item["output"] = cleaned_ma_text
#                 # Optionally refresh triples from dataset to be safe
#                 item["triples"] = data[sample_id - 1]
#                 fixed_ma += 1
#             except Exception as e:
#                 print(f"[MA][{lang}] Error re-running sample {sample_id}: {e}")
#                 # leave the old output as is

#     # Reindex to zero based
#     for pos, item in enumerate(multi_agent_results):
#         item["index"] = pos

#     print(f"[{lang}] Multi agent: fixed {fixed_ma} entries. Saving back to {ma_filename}...")
#     with open(ma_filename, "w", encoding="utf-8") as f:
#         json.dump(multi_agent_results, f, indent=4, ensure_ascii=False)

#     # ---------- E2E file ----------
#     print(f"[{lang}] Loading {e2e_filename}...")
#     with open(e2e_filename, "r", encoding="utf-8") as f:
#         e2e_results = json.load(f)

#     print(f"[{lang}] E2E entries: {len(e2e_results)}")

#     fixed_e2e = 0
#     for pos, item in enumerate(e2e_results):
#         original_index = item.get("index", pos)
#         sample_id = pos + 1

#         output = item.get("output", "")
#         if output in INVALID_OUTPUTS:
#             print(
#                 f"[E2E][{lang}] Re-running sample_id={sample_id} "
#                 f"(old index={original_index}) due to output='{output}'"
#             )
#             try:
#                 e2e_result = runner.run_end_to_end(
#                     sample_id=sample_id,
#                     provider=MODEL_PROVIDER,
#                     temperature=0.0,
#                 )
#                 raw_e2e_text = e2e_result.get("generated_text", "")
#                 cleaned_e2e_text = clean_text(raw_e2e_text)
#                 item["output"] = cleaned_e2e_text
#                 # Optionally refresh triples as well
#                 item["triples"] = data[sample_id - 1]
#                 fixed_e2e += 1
#             except Exception as e:
#                 print(f"[E2E][{lang}] Error re-running sample {sample_id}: {e}")
#                 # leave old output

#     # Reindex to zero based
#     for pos, item in enumerate(e2e_results):
#         item["index"] = pos

#     print(f"[{lang}] E2E: fixed {fixed_e2e} entries. Saving back to {e2e_filename}...")
#     with open(e2e_filename, "w", encoding="utf-8") as f:
#         json.dump(e2e_results, f, indent=4, ensure_ascii=False)


# def main():
#     # Run for both English and Irish
#     for lang in ["en", "ga"]:
#         repair_file_for_language(lang)

#     print("\nRepair and reindex completed for all languages.")


# if __name__ == "__main__":
#     main()


/home/chinonso/anaconda3/envs/lang2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-02 19:16:06.273998: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-02 19:16:06.301453: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-02 19:16:06.831911: I tensorflow/core/util/port.cc:153] oneDNN custom operatio


========== Language: EN ==========
Extracted 10 modified triplesets from data/D2T-1-FA_same3to6_min50_max500_sample.xml.
Building workflows with provider=openai, language=en.
Workflows built: ['default', 'unified_worker', 'single_module', 'no_guardrail', 'no_finalizer', 'no_orchestrator']
Dataset has 10 samples.
[en] Loading pilot_multi_agent_EN.json...
[en] Multi agent entries: 10
[MA][en] Re-running sample_id=9 (old index=9) due to output='NO SURFACE REALIZATION OUTPUT AVAILABLE.'
Running workflow='default' on sample_id=9 (index=8).


> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Final Answer",
  "action_input": [
    ["Washington,_D.C.", "type", "Capital_city"],
    ["Washington,_D.C.", "type", "Federal_district"],
    ["Washington,_D.C.", "country", "United_States"],
    ["United_States", "capital", "Washington,_D.C."],
    ["Union_(American_Civil_War)", "capital", "Washington,_D.C."],
    ["Pembina_Region", "capital", "Washington,_D.C."],
    ["Washington,_D.C.